# 05 - Baseline: clustering on the daily price only

Clusters the days using only the daily mean and standard deviation of the log real price, without scalograms or VAE. It is the reference every model has to beat. Like notebook 03, it only saves the regimes; every metric is computed afterwards by `scripts/06_regime_metrics.py`.

- `level_window = 1`: each day uses its own mean and standard deviation (daily baseline).
- `level_window = 7`: both are replaced by their backward 7-day mean, the day and the 6 days before it (7-day baseline, `tag = "7d"`).

The two columns are z-scored with training-day statistics, k-means is fitted on training days only and assigns every day, and regimes are numbered by ascending mean real price.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt
from IPython.display import display
from sklearn.manifold import TSNE

alt.data_transformers.enable('vegafusion')

from src.data import load_context, chronological_split
from src.clustering import with_level, level_columns
from src.plots import regime_timeline
from src.runs import prepare_runs, cluster_runs, tsne_grid, regime_strips
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

Set `level_window` and `tag` together: 1 and `""` for the daily baseline, 7 and `"7d"` for the 7-day baseline. `k_values` is always 3, 4 and 5.

In [ ]:
scalogram_set = "real_price_log"
k_values = [3, 4, 5]
study = "baseline"
level_window = 7
tag = "7d"

config = {
    "model": "level_only",
    "study": study,
    "tag": tag,
    "scalogram_set": scalogram_set,
    "k_values": k_values,
    "features": "daily mean and std of the log real price",
    "level_window": level_window,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "seed": 42,
}

## 1. Run folders and data

In [ ]:
config, run_dirs, run_configs, _ = prepare_runs(config)
for k, run_dir in run_dirs.items():
    print(f"k={k}: {exp.run_id(run_dir)}")

scalogram_dir = Path("../data/processed/scalograms") / config["scalogram_set"]
metadata = pd.read_csv(scalogram_dir / "scalogram_metadata.csv", parse_dates=["start_datetime", "end_datetime"])
context = load_context(Path("../data/processed"), metadata)
train_idx, val_idx, test_idx = chronological_split(len(metadata), config["train_ratio"], config["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]
print(exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx}))

level_cols = list(level_columns(metadata))
level = metadata.copy()
level[level_cols] = metadata[level_cols].rolling(config["level_window"], min_periods=1).mean()
features = with_level(np.zeros((len(metadata), 0)), level, train_idx)
print(f"Columns {level_cols}, backward mean over {config['level_window']} day(s), z-scored on training days")

## 2. Clustering

For each k, `cluster_runs` saves `regimes.csv`, `metrics.json`, the regime profile and clustering metrics tables, the t-SNE and the regime timeline, and the row in `registry.csv`.

In [ ]:
rng = np.random.default_rng(config["seed"])
plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
coords = TSNE(n_components=2, perplexity=30, random_state=config["seed"]).fit_transform(features[plot_idx])
results = cluster_runs(run_dirs, run_configs, features, context, train_idx, split_dates, coords, plot_idx)

## 3. Regimes

The t-SNE colored by the regimes of k = 3, 4 and 5, the regime timeline of each k and the regime strips together.

In [ ]:
display(tsne_grid(coords, plot_idx, results))
for k, ctx in results.items():
    display(regime_timeline(ctx, split_dates, title=f"Detected regimes over time (k={k})"))
display(regime_strips(results))